# Stage 3C - DeepSeek Thinking Chain Generation
## Model: `deepseek-ai/DeepSeek-R1-Distill-Qwen-14B`

This notebook runs DeepSeek-R1 on the **training set** using the 5-class improved prompt,
saves the `<think>...</think>` reasoning chains, then derives both 5-class and 2-class
training datasets from a single inference pass.

**Strategy:**
- Run 5-class inference once on `train_data.csv`
- Keep only correctly predicted samples (quality filter)
- Save `train_data_with_thinking_5class.csv`
- Derive `train_data_with_thinking_2class.csv` by mapping 1-2★→0, 4-5★→1, dropping 3★

**Output files** (saved to Drive):
- `LLM/deepseek_thinking_chains/train_data_with_thinking_5class.csv`
- `LLM/deepseek_thinking_chains/train_data_with_thinking_2class.csv`
- `LLM/deepseek_thinking_chains/generation_stats.json`

In [ ]:
!pip install -q vllm pandas scikit-learn tqdm

from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

In [ ]:
import os
import re
import json
import time
import numpy as np
import pandas as pd
from tqdm import tqdm

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer


In [ ]:
# ============================================
# Paths and configuration
# ============================================
TRAIN_PATH = "/content/drive/MyDrive/Yelp_Project/data/processed/train_data.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Yelp_Project/LLM/deepseek_thinking_chains"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TEXT_COL  = "text"
LABEL_COL = "stars"

MODEL_NAME    = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
BATCH_SIZE    = 8        # smaller batch for longer sequences
MAX_MODEL_LEN = 6144     # context length: input + thinking chain + answer
MAX_NEW_TOKENS = 1500    # enough for full thinking chain + answer

# Set N_SAMPLES to limit training data size and control runtime.
# Full dataset (39750) can take 15+ hours. 10000 takes ~5-7 hours on A100.
# Uses stratified sampling to keep class balance.
N_SAMPLES = 10000   # set to None to use the full training set

print(f"Model         : {MODEL_NAME}")
print(f"Train data    : {TRAIN_PATH}")
print(f"Output dir    : {OUTPUT_DIR}")
print(f"Max new tokens: {MAX_NEW_TOKENS}")
print(f"N_SAMPLES     : {N_SAMPLES} (None = full dataset)")


In [ ]:
# ============================================
# Load training set (with optional stratified sampling)
# ============================================
df_raw = pd.read_csv(TRAIN_PATH)[[TEXT_COL, LABEL_COL]].copy()
df_raw[LABEL_COL] = df_raw[LABEL_COL].astype(int)
df_raw["original_stars"] = df_raw[LABEL_COL]
df_raw["true_label_5class"] = df_raw[LABEL_COL]

if N_SAMPLES is not None and N_SAMPLES < len(df_raw):
    # Stratified sample: keep class distribution
    df_raw = (
        df_raw
        .groupby(LABEL_COL, group_keys=False)
        .apply(lambda g: g.sample(n=int(N_SAMPLES * len(g) / len(df_raw)), random_state=42))
        .reset_index(drop=True)
    )
    print(f"Sampled {len(df_raw)} training examples (stratified from full dataset)")
else:
    print(f"Using full training set: {len(df_raw)} samples")

print(df_raw["original_stars"].value_counts().sort_index())


In [ ]:
# ============================================
# Prompt builder using chat template
# KEY FIX: append <think> to force DeepSeek-R1 into thinking mode.
# ============================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

SYSTEM_PROMPT = " ".join([
    "You are a Yelp review star-rating predictor.",
    "Predict the star rating (1 to 5) for the following review.",
    "Output your final answer ONLY on the last line in this exact format:",
    "Final label: X",
    "where X is an integer from 1 to 5.",
])

def build_prompt(text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "Review: " + text},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    # Force DeepSeek-R1 to start thinking (chr(10) = newline)
    prompt = prompt + "<think>" + chr(10)
    return prompt

def parse_thinking_output(text: str):
    """
    Model output begins INSIDE <think> (we pre-filled <think>).
    Look for closing </think>, then extract Final label after it.
    """
    text = str(text).strip()
    close_match = re.search("(.*?)</think>" + r"\s*" + "(.*)", text, flags=re.DOTALL | re.IGNORECASE)
    if close_match:
        thinking_content = close_match.group(1).strip()
        after_think = close_match.group(2).strip()
    else:
        thinking_content = ""
        after_think = text

    pat = r"Final label:\s*([1-5])"
    label_matches = re.findall(pat, after_think, flags=re.IGNORECASE)
    if label_matches:
        return thinking_content, int(label_matches[-1]), True

    label_matches = re.findall(pat, text, flags=re.IGNORECASE)
    if label_matches:
        return thinking_content, int(label_matches[-1]), True

    star_matches = re.findall(r"([1-5])", after_think or text)
    if star_matches:
        return thinking_content, int(star_matches[-1]), False

    return thinking_content, None, False

# Sanity check
sample_prompt = build_prompt("Great food and excellent service!")
print("Prompt ends with:", repr(sample_prompt[-40:]))
print("Should end with: ...assistant\n<think>\n")


In [ ]:
# ============================================
# Load model
# skip_special_tokens=False: preserve <think> and </think> tags in output
# ============================================
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
    skip_special_tokens=False,   # CRITICAL: keep <think>...</think> tags
)

llm = LLM(
    model=MODEL_NAME,
    trust_remote_code=True,
    max_model_len=MAX_MODEL_LEN,
)
print("Model loaded.")


In [ ]:
# ============================================
# QUICK TEST: 200 samples to verify thinking chains are generated
# Run this cell BEFORE the full inference cell.
# If has_thinking_rate > 0, the fix is working — proceed to full run.
# ============================================
TEST_N = 200
test_batch = df_raw.sample(n=TEST_N, random_state=0).reset_index(drop=True)

test_prompts = [build_prompt(t) for t in test_batch[TEXT_COL].tolist()]
test_outputs = llm.generate(test_prompts, sampling_params)

test_rows = []
for i, out in enumerate(test_outputs):
    raw = out.outputs[0].text
    thinking, pred, strict = parse_thinking_output(raw)
    test_rows.append({"thinking_content": thinking, "pred": pred, "raw": raw})

test_df = pd.DataFrame(test_rows)
has_think = (test_df["thinking_content"].str.len() > 10).mean()

print(f"=== Quick Test Results ({TEST_N} samples) ===")
print(f"has_thinking_rate : {has_think:.3f}  (should be > 0)")
print(f"valid_pred_rate   : {test_df['pred'].notna().mean():.3f}")
print()
print("--- Sample 0 raw output (first 600 chars) ---")
print(test_df['raw'].iloc[0][:600])
print()
print("--- Sample 0 thinking_content (first 300 chars) ---")
print(str(test_df['thinking_content'].iloc[0])[:300])
print()
if has_think > 0:
    print("SUCCESS: thinking chains detected. Proceed to full inference cell.")
else:
    print("FAIL: no thinking chains found. Do NOT run full inference yet.")


In [ ]:
# ============================================
# Run inference on training set
# ============================================
all_rows = []
start_time = time.time()

for start in tqdm(range(0, len(df_raw), BATCH_SIZE), desc="Generating thinking chains"):
    end = min(start + BATCH_SIZE, len(df_raw))
    batch = df_raw.iloc[start:end]

    prompts = [build_prompt(text) for text in batch[TEXT_COL].tolist()]
    outputs = llm.generate(prompts, sampling_params)

    for i, output in enumerate(outputs):
        raw_text = output.outputs[0].text
        thinking_content, pred, strict_ok = parse_thinking_output(raw_text)
        true_label = int(batch.iloc[i]["true_label_5class"])

        all_rows.append({
            "text"             : batch.iloc[i][TEXT_COL],
            "original_stars"   : int(batch.iloc[i]["original_stars"]),
            "true_label_5class": true_label,
            "thinking_content" : thinking_content,
            "pred_5class"      : pred,
            "strict_format_ok" : strict_ok,
            "is_correct_5class": (pred == true_label) if pred is not None else False,
        })

elapsed = time.time() - start_time
result_df = pd.DataFrame(all_rows)

raw_path = os.path.join(OUTPUT_DIR, "train_raw_outputs.csv")
result_df.to_csv(raw_path, index=False, encoding="utf-8")

print(f"Inference done in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"Total samples    : {len(result_df)}")
print(f"Valid predictions: {result_df['pred_5class'].notna().sum()} ({result_df['pred_5class'].notna().mean()*100:.1f}%)")
print(f"Correct (5-class): {result_df['is_correct_5class'].sum()} ({result_df['is_correct_5class'].mean()*100:.1f}%)")
print(f"Has thinking chain: {(result_df['thinking_content'].str.len() > 0).sum()}")

In [ ]:
# ============================================
# DEBUG: Save 3 raw output texts to inspect
# ============================================
debug_rows = result_df.head(3)
for i, row in debug_rows.iterrows():
    print(f"=== Sample {i} | true={row['true_label_5class']} pred={row['pred_5class']} ===")
    print(f"thinking_content: {repr(str(row['thinking_content'])[:300])}")
    print()


In [ ]:
# ============================================
# Build 5-class training dataset
# Quality filter: correct prediction + non-empty thinking chain
# ============================================
df_5class = result_df[
    result_df["is_correct_5class"] &
    (result_df["thinking_content"].str.len() > 10)
].copy()

df_5class = df_5class[[
    "text", "original_stars", "true_label_5class", "thinking_content"
]].rename(columns={"true_label_5class": "label"})

out_5class = os.path.join(OUTPUT_DIR, "train_data_with_thinking_5class.csv")
df_5class.to_csv(out_5class, index=False, encoding="utf-8")

print(f"5-class training samples: {len(df_5class)} / {len(result_df)} ({len(df_5class)/len(result_df)*100:.1f}% kept)")
print("Label distribution:")
print(df_5class["label"].value_counts().sort_index())
print(f"Saved to: {out_5class}")

In [ ]:
# ============================================
# Derive 2-class training dataset from 5-class results
# Map: 1-2★ → 0 (Negative), 4-5★ → 1 (Positive), drop 3★
# ============================================
df_2class_base = result_df[
    result_df["is_correct_5class"] &
    (result_df["thinking_content"].str.len() > 10) &
    (result_df["original_stars"] != 3)   # drop 3-star
].copy()

df_2class_base["label"] = df_2class_base["original_stars"].apply(
    lambda x: 0 if x <= 2 else 1
)
df_2class_base["pred_2class"] = df_2class_base["pred_5class"].apply(
    lambda x: 0 if x is not None and x <= 2 else (1 if x is not None and x >= 4 else None)
)
df_2class_base["is_correct_2class"] = df_2class_base["label"] == df_2class_base["pred_2class"]

# Only keep samples where 2-class mapping is also correct
df_2class = df_2class_base[df_2class_base["is_correct_2class"]].copy()
df_2class = df_2class[["text", "original_stars", "label", "thinking_content"]]

out_2class = os.path.join(OUTPUT_DIR, "train_data_with_thinking_2class.csv")
df_2class.to_csv(out_2class, index=False, encoding="utf-8")

print(f"2-class training samples: {len(df_2class)} / {len(result_df[result_df['original_stars'] != 3])} ({len(df_2class)/len(result_df[result_df['original_stars'] != 3])*100:.1f}% kept)")
print("Label distribution:")
print(df_2class["label"].value_counts().sort_index())
print(f"Saved to: {out_2class}")

In [ ]:
# ============================================
# Save generation statistics
# ============================================
stats = {
    "model"                   : MODEL_NAME,
    "total_train_samples"     : len(result_df),
    "valid_pred_rate"         : float(result_df["pred_5class"].notna().mean()),
    "correct_5class_rate"     : float(result_df["is_correct_5class"].mean()),
    "has_thinking_rate"       : float((result_df["thinking_content"].str.len() > 10).mean()),
    "kept_5class"             : int(len(df_5class)),
    "kept_2class"             : int(len(df_2class)),
    "generation_time_s"       : float(elapsed),
    "avg_thinking_length"     : float(df_5class["thinking_content"].str.len().mean()),
}

stats_path = os.path.join(OUTPUT_DIR, "generation_stats.json")
with open(stats_path, "w") as f:
    json.dump(stats, f, indent=2)

print("=== Generation Summary ===")
for k, v in stats.items():
    print(f"  {k}: {v}")